# LiveMIEL Pipeline

The LiveMIEL pipeline is an automated image-processing and computational biology workflow designed for microenvironment imaging embedding analysis of single nuclei. Original author: Alexandra Shuvaeva 

The pipeline includes the following analytical blocks:

* **Tuning parameters for image segmentation:** lowSigm, highSigm, thresh and FalsePositBrightness_k parameters should be tuned for further segmentation and cropping.
* **Nucleus Segmentation & Cropping:** Automated localization and masking of individual cellular nuclei, followed by image extraction to generate single-nuclei image datasets.
* **Feature Extraction:** Quantitative profiling of nuclear morphometrics, textures, and spatial distributions from single-nuclei images, including:
  - *Texture Features:* Haralick texture features (GLCM) and Threshold Adjacency Statistics (TAS).
  - *Shape & Geometry:* Zernike moments and Center of Mass (centroid tracking).
  - *Epigenetic/Chromatin Architecture:* Higher-order chromatin intensity features and signal distribution patterns.
* **Dimensionality Reduction:** Principal Component Analysis (PCA) to reduce the high-dimensional feature vectors while preserving maximum biological variance.
* **Clustering:** Unsupervised clustering of the obtained feature vectors to identify distinct nuclear/epigenetic states across the differentiation timeline.
* **Downstream Analysis:** Statistical evaluation and structural profiling of the resulting cluster data distributions.

# Getting Started

Follow these steps to set up your local Python environment and install all required dependencies.

### 1. Environment Setup

We recommend using **Conda** to manage an isolated environment with **Python 3.10**:

```bash
# Create a new conda environment
conda create -n livemiel python=3.10 pip -y

# Activate the environment
conda activate livemiel
```

### 2. Installing Dependencies

Install the core image processing, machine learning, and data analysis libraries via `pip`:

```bash
pip install mahotas scikit-image matplotlib scikit-learn opencv-python tqdm pandas pingouin statannotations

```

### 3. Installing PyTorch

Depending on your hardware configuration (CPU vs GPU), choose **one** of the following options to install PyTorch:

* **Option A: Standard PyTorch installation via Conda**
  ```bash
  conda install pytorch torchvision torchaudio -c pytorch
  ```

* **Option B: PyTorch with CUDA 12.1 support via Pip (Recommended for NVIDIA GPUs)**
  ```bash
  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
  ```
  *Note: Make sure to modify the CUDA version (`cu121`) to match your physical GPU drivers.*


In [ ]:
# to check your current torch version
import torch
print(torch.__version__)

### Importing libraries

In [ ]:
# importing libraries

import numpy as np
import os
from skimage import io
FOLDERNAME = 'C:/Users/User/Desktop/LiveMIEL/' # Your local foldername with required files:
                            #bandpass_segmentation.py
                            #cropping_single_cells.py
                            #features_extraction.py
                            #classification.py
import sys

sys.path.append('C:/Users/User/Desktop/LiveMIEL/{}'.format(FOLDERNAME)) # Your local foldername
import bandpass_segmentation as bs
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import ListedColormap
from sklearn import metrics

import classification
from features_extraction import HaralickFeatures, TAS, \
ZernikeMoments, extract, Preprocess, CenterMass, ChromatinFeatures
from sklearn.model_selection import train_test_split, cross_val_score

In [ ]:
# to check your current location if needed
pwd

# Nuclei segmentation

Enter the directory path of the nuclei images for segmentation:

In [ ]:
# choose your main directory here
dir = './IPSC/nuclei_images/AF9-Red_NGN2/2d/'

In [ ]:
lst = os.listdir((dir))
lst.sort()

save_dir_nuclei = path = dir.replace('nuclei_images', 'single_nuclei_images', 1)

try:
    os.makedirs(path, exist_ok = True)
    print("Save_dir_nuclei created successfully")
except OSError as error:
    print("Save_dir_nuclei can not be created")

save_dir_info = path = dir.replace('nuclei_images', 'saved_info_single_cells', 1)

try:
    os.makedirs(path, exist_ok = True)
    print("Save_dir_nuclei created successfully")
except OSError as error:
    print("Save_dir_nuclei can not be created")

## Tuning parameters for image segmentation

Choose image for segmentation parameters tuning:

In [ ]:
# Enter image name to tunining segmentation parameters on
image_name = str('Exp_96.TIF')

### Tuning Parameters for Image Segmentation

To achieve optimal segmentation performance on current microscopy dataset, you can tune the core parameters in the execution line:

```python
image_segm = bs.main(image, lowSigm=10, highSigm=70, coeff=1, thresh=0.003, FalsePositBrightness_k=2)
```

#### Parameter Breakdown & Optimization Tips:

| Parameter | Recommended Range | Primary Function & Biological Context |
| :--- | :--- | :--- |
| `lowSigm` | `5` – `12` *(Up to 40)* | **Noise Reduction & Smoothing.** Higher values prevent a single overlapping or irregular nucleus from being falsely split into multiple distinct objects. *Note: Must always be strictly lower than `highSigm`.* |
| `highSigm` | `30` – `70` | **Background Removal.** Controls the upper limit of the gaussian filter. Setting `highSigm` lower (e.g., around `40`) significantly speeds up the code execution. |
| `thresh` | `0.003` – `0.05` | **Intensity Threshold.** Eliminates background image areas where the fluorescence intensity falls below the specified absolute value. |
| `FalsePositBrightness_k` | `1.2` – `2.5` | **False Positive Filtering.** Post-segmentation coefficient to discard artifacts and dim/dying nuclei with low fluorescence intensity from the final dataset. |
| `coeff` | `1` | **Scaling Coefficient.** Internal multiplier for spatial adjustment (default is set to `1`). |

*Quick Validation Presets:* 
* For fast initial testing and validation, the combinations `lowSigm=15 / highSigm=40` or `lowSigm=10 / highSigm=70` generally yield the most stable baseline results.


In [ ]:
print(os.path.join(dir, image_name))
image = io.imread(os.path.join(dir, image_name))
image_segm = bs.main(image, lowSigm=40, highSigm=70, coeff=1, thresh=0.003, FalsePositBrightness_k = 2) #tune parameters here for better segmentation performance

jet = cm.get_cmap('jet', 256)
newcolors = jet(np.linspace(0, 1, np.max(image_segm)))
black = np.array([0, 0, 0, 1])
newcolors[:1, :] = black
newcmp = ListedColormap(newcolors)

fig, ax = plt.subplots(1,2, figsize=(50, 20))
ax[0].axis('off')
ax[0].imshow(image, cmap='gray')
ax[0].set_title('Original image (in RGB)', fontsize=40)
ax[1].axis('Off')
ax[1].imshow(image_segm, cmap=newcmp)
ax[1].set_title('Segmented and labeled image', fontsize=40)
fig.subplots_adjust(wspace=0.02, hspace=0,
                                top=0.9, bottom=0.05, left=0, right=0.75)

## Segmentation + cropping images into single cells

In [ ]:
import cropping_single_cells as csc

raw_img_names = np.empty(0)
number_segmented = np.empty(0, dtype=int)
labels = np.empty(0, dtype=int)
coordinates = np.empty((0, 4), dtype=int)

for i, img in enumerate(lst):
    # print(os.path.join(dir, img))
    image = io.imread(os.path.join(dir, img))
    image_segm = bs.main(image, lowSigm=10, highSigm=70, coeff=1, thresh=0.03, FalsePositBrightness_k = 2) #put tuned parameters for segmentation

    crops, num_segm, l, coord = csc.cropping_cells(image_segm, image, save_dir_nuclei, img, save=True)

    raw_img_names = np.append(raw_img_names, img)
    number_segmented = np.append(number_segmented, num_segm)
    labels = np.append(labels, l)
    coordinates = np.append(coordinates, coord, axis=0)

    np.save(os.path.join(save_dir_info, 'raw_img_names.npy'), raw_img_names)
    np.save(os.path.join(save_dir_info, 'number_segmented.npy'), number_segmented)
    np.save(os.path.join(save_dir_info, 'labels.npy'), labels)
    np.save(os.path.join(save_dir_info, 'coordinates.npy'), coordinates)

# Collecting features from the segmented images

## Features extraction

Path to cropped single-nuclei dataset:

In [ ]:
# Using all days of differentiation for the AF9-Red sensor as an example

directories = ['./IPSC/single_nuclei_images/AF9-Red_NGN2/0d/',
               './IPSC/single_nuclei_images/AF9-Red_NGN2/1d/',
                './IPSC/single_nuclei_images/AF9-Red_NGN2/2d/',
                './IPSC/single_nuclei_images/AF9-Red_NGN2/3d/',
               './IPSC/single_nuclei_images/AF9-Red_NGN2/4d/']

Selected features to use in classification:
  

*   Haralick features

*   TAS features
*   Zernike moments - if **None**, then radius is calculated automatically
*   Center of mass of each nucleus
*   Chromatin features - minimum area of a chromatin "dot", maximum area of a chromatin "dot", mean area of a chromatin "dot", total area of all chromatin "dots"
*   feat_num - number of each selected features
*   objects_ - number of the nuclei to classify in each class
*   object_class - names of the cell families








In [ ]:
features = {'features': [HaralickFeatures(), TAS(), ZernikeMoments(None), CenterMass(), ChromatinFeatures()],
            'feat_num' : [13, 54, 25, 2, 4]}

for i in range(len(directories)):
    obj_key = 'objects_' + str(i)
    obj_value = len(os.listdir(directories[i]))
    class_key = 'object_' + str(i) + '_class'
    class_value = str(input("Object class for \n%s \n" %directories[i]))
    features[obj_key] = obj_value
    features[class_key] = class_value

preproc_commands = ['meanvar', 'pad'] # can be ['meanvar', 'pad'] or one of them, or empty

In [ ]:
df, X, y = \
classification.dataCollection(directories, features, preproc_commands, normalize=True, enhance=True)
# if normalize=True -- images are normalized to 0-255 prior to fetures extraction
# if enhance=True -- images' brightnesses are normalized to common brightness=0.09

## Features averaging

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

y_arr = np.array(list(y))

print(y_arr, y_arr.shape)

Features averaged over all nuclei within original images

In [ ]:
import re

num_sorted = np.empty(0, dtype=int)
for i in range(len(directories)):
    lst = os.listdir((directories[i]))
    lst.sort()

    num_segm = np.empty(0, dtype=int)

    ref_img = re.search('(.*)_', lst[0]).group() #extraction of original microscopic image name from single nuclei image name
    counter = 0

    for img in lst:
        cur_raw_img = re.search('(.*)_', img).group() #extraction of original microscopic image name from single nuclei image name
        print(ref_img, cur_raw_img)
        if cur_raw_img == ref_img:
            counter += 1
        if cur_raw_img > ref_img:
            num_segm = np.append(num_segm, counter)
            ref_img = cur_raw_img
            counter = 1
    num_segm = np.append(num_segm, counter)
    num_sorted = np.append(num_sorted, num_segm)

In [ ]:
features_num = sum(features['feat_num'])

center_vector = []
y_names = np.empty(0, dtype = np.dtype('U25'))

X_y = np.append(X_std, y_arr.reshape(len(y_arr), 1), axis = 1)
X_y = X_y[X_y[:, features_num].argsort()]

start_r = 0
stop_r = 0

for i in range(len(num_sorted)):
  stop_r += num_sorted[i]
  center_vector.append((np.mean(np.asarray((X_y[start_r : stop_r, :features_num]), dtype='float64', order='C'), axis = 0)))
  y_names = np.append(y_names, X_y[start_r][features_num])
  start_r = stop_r

X = center_vector
y = y_names

Features averaged over N nuclei within image class

In [ ]:
N = 60 # averaging over 60 nuclei was considered to be basic
#N_set = [20, 40, 40, 40, 40] - N value for each nuclei class

In [ ]:
features_num = np.sum(features['feat_num'])

center_vector = []
y_names = np.empty(0, dtype = np.dtype('U25'))

X_y = np.append(X_std, y_arr.reshape(len(y_arr), 1), axis = 1)
X_y = X_y[X_y[:, features_num].argsort()]

start_r = 0
stop_r = 0

for i in range(len(directories)):
  #N = N_set[i]
  for j in range(len(os.listdir(directories[i]))//N):
    stop_r += N
    center_vector.append((np.mean(np.asarray((X_y[start_r : stop_r, :features_num]), dtype='float64', order='C'), axis = 0)))
    y_names = np.append(y_names, X_y[start_r][features_num])
    start_r = stop_r

  if len(os.listdir(directories[i]))/N == len(os.listdir(directories[i]))//N:
    continue
  else:
    stop_r += len(os.listdir(directories[i])) % N
    center_vector.append((np.mean(np.asarray((X_y[start_r : stop_r, :features_num]), dtype='float64', order='C'), axis = 0)))
    y_names = np.append(y_names, X_y[start_r][features_num])
    start_r = stop_r

X = center_vector
y = y_names

## Features preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler1 = StandardScaler()

In [ ]:
X_std = scaler1.fit_transform(X)

print(np.sum(np.double(np.isnan(X))) / np.prod(np.shape(X)))
print(X_std)

# PCA

## Dataset preprocessing

In [ ]:
labels = np.reshape(y,(y.shape[0],1))
final_data = np.concatenate([X_std, labels],axis=1)

In [ ]:
feature_names = np.empty(0)

for i in range(13):
    feature_names = np.append(feature_names, 'Haralick features ' + str(i+1))

for i in range(13, 67):
    feature_names = np.append(feature_names, 'TAS ' + str(i-12))

for i in range(67, 92):
    feature_names = np.append(feature_names, 'Zernike moments ' + str(i-66))

feature_names = np.append(feature_names, 'Center of mass (x)')
feature_names = np.append(feature_names, 'Center of mass (y)')
feature_names = np.append(feature_names, 'Chromatin features (min)')
feature_names = np.append(feature_names, 'Chromatin features (max)')
feature_names = np.append(feature_names, 'Chromatin features (mean)')
feature_names = np.append(feature_names, 'Chromatin features (total)')

In [ ]:
import pandas as pd

cur_dataset = pd.DataFrame(final_data)
features_classes = feature_names
features_labels = np.append(features_classes,'label')
cur_dataset.columns = features_labels

## Explained variance for PCA

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components = 10)

X_train_pca = pca.fit_transform(X_std)

exp_var_pca = pca.explained_variance_ratio_

cum_sum_eigenvalues = np.cumsum(exp_var_pca)

for i in range(3):
    print(
        "For PC",
        i+1,
        "the explained variance is :",
        exp_var_pca[i],
    )

plt.bar(range(0,len(exp_var_pca)), exp_var_pca, alpha=0.5, align='center', label='Individual explained variance')
plt.step(range(0,len(cum_sum_eigenvalues)), cum_sum_eigenvalues, where='mid',label='Cumulative explained variance')
plt.ylabel('Explained variance ratio')
plt.xlabel('Principal component index')
plt.legend(loc='center right')
plt.tight_layout()
plt.show()

# PCA plots

### Interactive 3D PCA plot

In [ ]:
import plotly.express as px

from sklearn.decomposition import PCA
pca_data = PCA(n_components = 3)
principalComponents = pca_data.fit_transform(X_std)

principal_Df = pd.DataFrame(data = principalComponents
             , columns = ['principal component 1', 'principal component 2', 'principal component 3'])

import seaborn as sns

fig_3d = px.scatter_3d(
    principalComponents, x=0, y=1, z=2,
    color=y_names, labels={'color': 'days'})

fig_3d.update_traces(marker_size=5)

fig_3d.show()

# Choose the file name here:
fig_3d.write_html("AF9-Red.html")

### 2D PCA plot

In [ ]:
from sklearn.decomposition import PCA
pca_data = PCA(n_components = 3)
principalComponents = pca_data.fit_transform(X_std)

principal_Df = pd.DataFrame(data = principalComponents
             , columns = ['principal component 1', 'principal component 2', 'principal component 3'])

plt.figure()
plt.figure(figsize=(10,10))
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel('Principal Component - 1',fontsize=20)
plt.ylabel('Principal Component - 2',fontsize=20)
plt.title("PCA of IPSC Dataset - 0-4 days \n (feature values averaged over 60 nuclei)",fontsize=20)
targets = ['Day 0', 'Day 1', 'Day 2', 'Day 3', 'Day 4']
colors = ['#6F518F', '#549EAC', '#B8BC3D', '#D0843E', '#AD2C24']

for target, color in zip(targets,colors):
    indicesToKeep = cur_dataset['label'] == target
    plt.scatter(principal_Df.loc[indicesToKeep, 'principal component 1'],
                principal_Df.loc[indicesToKeep, 'principal component 2'], c = color, s = 80, edgecolors='k')

plt.legend(targets,prop={'size': 15})

for target, color in zip(targets,colors):
    indicesToKeep = cur_dataset['label'] == target
    plt.scatter(np.mean(principal_Df.loc[indicesToKeep, 'principal component 1']),
                np.mean(principal_Df.loc[indicesToKeep, 'principal component 2']), s=350, marker="D", c=color,
                lw=2.5, edgecolors="black")

# Clustering

## Hopkins statistics

In [ ]:
X = principal_Df[['principal component 1', 'principal component 2']]
X = X.dropna(axis=0)

In [ ]:
from sklearn.neighbors import NearestNeighbors
from random import sample
from numpy.random import uniform
from math import isnan
def hopkins(X):
  d = X.shape[1]
  n = len(X)
  m = int(0.1 * n)
  nbrs = NearestNeighbors(n_neighbors=1).fit(X.values)

  rand_X = sample(range(0, n, 1), m)

  ujd = []
  wjd = []
  for j in range(0, m):
     u_dist, _ = nbrs.kneighbors(uniform(np.amin(X,axis=0),np.amax(X,axis=0),d).reshape(1, -1), 2, return_distance=True)
     ujd.append(u_dist[0][1])
     w_dist, _ = nbrs.kneighbors(X.iloc[rand_X[j]].values.reshape(1, -1), 2, return_distance=True)
     wjd.append(w_dist[0][1])

  H = sum(wjd) / (sum(ujd) + sum(wjd))
  if isnan(H):
     print(ujd, wjd)
     H = 0

  return H

Hopkins statistics (H):

*   highly clustered data - H ~ 0
*   clustered data - H < 0.5  
*   random data - H ~ 0.5
*   regular / uniform data - H ~ 1.

In [ ]:
hopkins(X)

### Сomparing Hopkins statistics between sensors (optional)

In [ ]:
# An example for AF9-Red sensor:
hopkins_AF9Red = np.array([])
for i in range(100):
    new_hopkins = np.array([hopkins(X)])
    hopkins_AF9Red = np.concatenate((hopkins_AF9Red, new_hopkins), axis=0)
print(hopkins_AF9Red, hopkins_AF9Red.shape)

# To save obtained dataset of 100-times calculated Hopkins statistics for the AF9-Red sensor
np.save('AF9-Red-hopkins-data.npy', hopkins_AF9Red)

a = np.load('AF9-Red-hopkins-data.npy')
average_af9 = np.mean(a)
print(average_af9)
std_af9 = np.std(a)
print(std_af9)

# This node must be performed for all 3 sensors for the further comparison

In [ ]:
from scipy.stats import t

# Step 1: Calculate the T-score (assumes normal distribution of data)
def calculate_t_score(sample1, sample2):
    mean1 = np.mean(sample1)
    mean2 = np.mean(sample2)
    std1 = np.std(sample1, ddof=1) 
    std2 = np.std(sample2, ddof=1)
    n1 = len(sample1)
    n2 = len(sample2)
 
    t_score = (mean1 - mean2) / np.sqrt((std1**2 / n1) + (std2**2 / n2))
    return t_score
 
# Step 2: Determine the degrees of freedom (df)
def calculate_degrees_of_freedom(sample1, sample2):
    n1 = len(sample1)
    n2 = len(sample2)
    df = n1 + n2 - 2  # For a two-sample t-test
    return df
 
# Step 3: Identify the appropriate t-distribution
# (The scipy.stats.t distribution is used, which automatically considers the degrees of freedom)
 
# Step 4: Find the p-value
def calculate_p_value(t_score, df):
    p_value = 2 * (1 - t.cdf(np.abs(t_score), df))
    return p_value
 
# Step 5: Interpret the p-value
def interpret_p_value(p_value, alpha=0.05):
    if p_value < alpha:
        return "Reject the null hypothesis. There is a statistically significant difference."
    else:
        return "Fail to reject the null hypothesis. There is no statistically significant difference."
    
# Step 5: Calculate Cohen's d
def calculate_cohens_d(sample1, sample2):
    n1, n2 = len(sample1), len(sample2)
    mean1, mean2 = np.mean(sample1), np.mean(sample2)
    # ddof=1 for unbiased variance estimation
    var1, var2 = np.var(sample1, ddof=1), np.var(sample2, ddof=1)
    
    # Calculating pooled SD
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    # The magnitude of the effect (we take the module to evaluate the force, not the direction)
    d = np.abs(mean1 - mean2) / pooled_std
    return d

def interpret_cohens_d(d):
    if d < 0.2:   return "Negligible effect"
    elif d < 0.5: return "Small effect"
    elif d < 0.8: return "Medium effect"
    else:         return "Large effect"

# To read more about T-score: geeksforgeeks.org/how-to-find-a-p-value-from-a-t-score-in-python/

In [ ]:
# Pair-wise statistical comparison of MPP8-Green and AF9-Red sensors
sample1 = pwwp_npy
sample2 = af9_npy

t_score = calculate_t_score(sample1, sample2)
df = calculate_degrees_of_freedom(sample1, sample2)
p_value = calculate_p_value(t_score, df)
result = interpret_p_value(p_value)

cohens_d = calculate_cohens_d(sample1, sample2)
effect_size_label = interpret_cohens_d(cohens_d)
 
print("p-value:", p_value)
print(result)
print(f"Cohen's d: {cohens_d:.4f} ({effect_size_label})")

Plotting histogram:

In [ ]:
import matplotlib as mtl
from pylab import rcParams

mtl.rcParams['legend.frameon'] = 'False' #'True'
rcParams['figure.figsize'] = 7, 7
rcParams['font.size'] = 14

af9_npy = np.load('AF9-Red-hopkins-data.npy')
pwwp_npy = np.load('PWWP-Red-hopkins-data.npy')
mpp8_npy = np.load('MPP8-Green-hopkins-data.npy')

average_af9 = np.mean(af9_npy)
average_pwwp = np.mean(pwwp_npy)
average_mpp8 = np.mean(mpp8_npy)

std_af9 = np.std(af9_npy)
std_pwwp = np.std(pwwp_npy)
std_mpp8 = np.std(mpp8_npy)

data = np.array([[average_af9, std_af9],
[average_pwwp, std_pwwp],
[average_mpp8, std_mpp8]])

fig, ax = plt.subplots()

labelsx = ['AF9-Katushka', 'PWWP-Katushka', 'MPP8-Green']
x = np.arange(len(labelsx))  # the label locations
plt.xticks(x, labelsx)
# tilt of the x labels
plt.xticks(fontsize=20, weight='bold', fontname='arial', rotation=10)
plt.yticks(fontsize=20, weight='bold', fontname='arial')

width = 0.6  # the width of the bars

colors = ['black', 'darkgray', 'dimgray']
labels_hist = ['AF9-Katushka','PWWP-Katushka','MPP8-Green']
hatches = [None, None, None]

rects1 = ax.bar(x, data[:,0], width, color=colors[:], edgecolor=None, label=labels_hist[:], linewidth=1.5, hatch = hatches[:])
    
plt.title('Hopkins statistics for clustered data quality', fontsize=20, weight='bold', fontname='arial', pad=40)

plt.ylabel('Hopkins statistics (H)', fontsize=20, fontname='arial', weight='bold')
plt.xlabel("Sensor imaging datasets", fontsize=20, fontname='arial', weight='bold')

plt.ylim(0, 0.5)

for i in range(len(labelsx)):
    plt.errorbar( i, data[i,0], yerr = data[i,1], ecolor='black', capsize=10, elinewidth=5, markeredgewidth=5)

# statistical annotation
def draw_brace(ax, xspan, yy, text):
    """Draws an annotated brace on the axes."""
    xmin, xmax = xspan
    xspan = xmax - xmin
    ax_xmin, ax_xmax = ax.get_xlim()
    xax_span = ax_xmax - ax_xmin

    ymin, ymax = ax.get_ylim()
    yspan = ymax - ymin
    resolution = int(xspan/xax_span*100)*2+1 # guaranteed uneven
    beta = 100./xax_span # the higher this parameter is, the smaller the radius

    x = np.linspace(xmin, xmax, resolution)
    x_half = x[:int(resolution/2)+1]
    y_half_brace = (1/(1.+np.exp(-beta*(x_half-x_half[0])))
                    + 1/(1.+np.exp(-beta*(x_half-x_half[-1]))))
    y = np.concatenate((y_half_brace, y_half_brace[-2::-1]))
    y = yy + (.05*y - .01)*yspan # adjust vertical position

    ax.autoscale(False)
    ax.plot(x, y, color='white', lw=1)

    ax.text((xmax+xmin)/2., yy+.07*yspan, text, ha='center', va='bottom', size=36, fontname='arial')

import matplotlib.patches as mpatches

ax.annotate("",
            xy=(0, 0.43), xycoords='data',
            xytext=(2, 0.43), textcoords='data',
            arrowprops=dict(arrowstyle="-", connectionstyle="arc3", linewidth=4))
ax.annotate("",
            xy=(0, 0.38), xycoords='data',
            xytext=(1, 0.38), textcoords='data',
            arrowprops=dict(arrowstyle="-", connectionstyle="arc3", linewidth=4))
ax.annotate("",
            xy=(1, 0.33), xycoords='data',
            xytext=(2, 0.33), textcoords='data',
            arrowprops=dict(arrowstyle="-", connectionstyle="arc3", linewidth=4))

draw_brace(ax, (0, 2), 0.39, '***')
draw_brace(ax, (0, 1), 0.34, '****')
draw_brace(ax, (1, 2), 0.29, '****')

sns.despine(fig=None, ax=None, top=True, right=True, left=False, bottom=False, offset=None, trim=False)

plt.text(0.5, 0.7, '')

# If you want to save your figure
plt.savefig(fname='hopkins_of_all.tiff', dpi='figure', format='tiff',
        facecolor='white', edgecolor='auto'
       )

ax.spines['left'].set_linewidth(3)
ax.spines['bottom'].set_linewidth(3)

plt.show()

## Silhouette coefficient analysis

### For K-means clustering

In [ ]:
X = principalComponents

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score

import matplotlib.cm as cm

range_n_clusters = [2, 3, 4, 5, 6]

for n_clusters in range_n_clusters:
    # Create a subplot with 1 row and 2 columns
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_size_inches(18, 7)

    # The 1st subplot is the silhouette plot
    # The silhouette coefficient can range from -1, 1
    ax1.set_xlim([-0.1, 1])
    # The (n_clusters+1)*10 is for inserting blank space between silhouette
    # plots of individual clusters, to demarcate them clearly.
    ax1.set_ylim([0, len(X) + (n_clusters + 1) * 10])

    # Initialize the clusterer with n_clusters value and a random generator
    # seed of 10 for reproducibility.
    clusterer = KMeans(n_clusters=n_clusters, n_init=1, random_state=10)
    cluster_labels = clusterer.fit_predict(X)

    # The silhouette_score gives the average value for all the samples.
    silhouette_avg = silhouette_score(X, cluster_labels)
    print(
        "For n_clusters =",
        n_clusters,
        "The average silhouette_score is :",
        silhouette_avg,
    )

    # Compute the silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(X, cluster_labels)

    y_lower = 10
    for i in range(n_clusters):
        # Aggregate the silhouette scores for samples belonging to
        # cluster i, and sort them
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]

        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )

        # Label the silhouette plots with their cluster numbers at the middle
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

        # Compute the new y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    ax1.set_title("The silhouette plot for the various clusters.")
    ax1.set_xlabel("The silhouette coefficient values")
    ax1.set_ylabel("Cluster label")

    # The vertical line for average silhouette score of all the values
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")

    ax1.set_yticks([])  # Clear the yaxis labels / ticks
    ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

    # 2nd Plot showing the actual clusters formed
    colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
    ax2.scatter(
        X[:, 0], X[:, 1], marker=".", s=300, lw=0, alpha=0.7, c=colors, edgecolor="k"
    )

    # Labeling the clusters
    centers = clusterer.cluster_centers_
    # Draw white circles at cluster centers
    ax2.scatter(
        centers[:, 0],
        centers[:, 1],
        marker="o",
        c="white",
        alpha=1,
        s=200,
        edgecolor="k",
    )

    for i, c in enumerate(centers):
        ax2.scatter(c[0], c[1], marker="$%d$" % i, alpha=1, s=50, edgecolor="k")

    ax2.set_title("The visualization of the clustered data.")
    ax2.set_xlabel("PC1")
    ax2.set_ylabel("PC2")

    plt.suptitle(
        "Silhouette analysis for KMeans clustering on sample data with n_clusters = %d"
        % n_clusters,
        fontsize=14,
        fontweight="bold",
    )

plt.show()